# Kernel Benchmark Comparison

This notebook compares runtime and peak CUDA memory usage for the four kernel implementations in `kernels/` across the requested input shapes.


In [ ]:
from pathlib import Path
import statistics
import sys
import time

import torch

repo_root = Path.cwd()
if repo_root.name == "benchmarks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from kernels.optimized_pope import pope_fwd_attention as optimized_pope_attention
from kernels.rope import rope_freqs, simple_rope_attention
from kernels.simple_flash_attention import fwd_attention
from kernels.simple_pope import pope_fwd_attention as simple_pope_attention

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required to run these benchmarks.")

torch.manual_seed(0)
torch.set_grad_enabled(False)

DEVICE = "cuda"
DTYPE = torch.float16
SHAPES = [
    (1, 8, 512, 64),
    (1, 8, 1024, 64),
    (1, 8, 2048, 64),
    (1, 8, 4096, 64),
    (2, 8, 1024, 64),
    (4, 8, 1024, 64),
]
WARMUP = 3
REP = 7
MEMORY_TRIALS = 1


In [ ]:
def flash_impl(q, k, v, sm_scale):
    return fwd_attention(q, k, v, sm_scale)


def rope_impl(q, k, v, sm_scale):
    cos, sin = rope_freqs(q.shape[-2], q.shape[-1], device=q.device)
    return simple_rope_attention(q, k, v, cos, sin, sm_scale)


def simple_pope_impl(q, k, v, sm_scale):
    return simple_pope_attention(q, k, v, sm_scale)


def optimized_pope_impl(q, k, v, sm_scale):
    return optimized_pope_attention(q, k, v, sm_scale)


IMPLEMENTATIONS = {
    "flash": flash_impl,
    "rope": rope_impl,
    "simple_pope": simple_pope_impl,
    "optimized_pope": optimized_pope_impl,
}


def make_inputs(batch, heads, seq_len, dim, dtype=DTYPE, device=DEVICE):
    shape = (batch, heads, seq_len, dim)
    q = torch.randn(shape, device=device, dtype=dtype)
    k = torch.randn(shape, device=device, dtype=dtype)
    v = torch.randn(shape, device=device, dtype=dtype)
    sm_scale = dim ** -0.5
    return q, k, v, sm_scale


def benchmark_runtime(fn, q, k, v, sm_scale, warmup=WARMUP, rep=REP):
    for _ in range(warmup):
        _ = fn(q, k, v, sm_scale)
    torch.cuda.synchronize()

    timings_ms = []
    for _ in range(rep):
        start = time.perf_counter()
        _ = fn(q, k, v, sm_scale)
        torch.cuda.synchronize()
        timings_ms.append((time.perf_counter() - start) * 1000.0)
    return statistics.median(timings_ms)


def benchmark_peak_memory(fn, q, k, v, sm_scale, trials=MEMORY_TRIALS):
    peaks_mb = []
    for _ in range(trials):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        baseline = torch.cuda.memory_allocated()
        out = fn(q, k, v, sm_scale)
        torch.cuda.synchronize()
        peak = torch.cuda.max_memory_allocated()
        peaks_mb.append((peak - baseline) / (1024 ** 2))
        del out
    return max(peaks_mb)


def shape_label(row):
    return f"{row['B']}x{row['H']}x{row['S']}x{row['D']}"


def print_metric_table(rows, metric_key, metric_label, value_format):
    shapes = []
    for row in rows:
        label = shape_label(row)
        if label not in shapes:
            shapes.append(label)

    implementations = list(IMPLEMENTATIONS)
    by_impl = {name: {} for name in implementations}
    for row in rows:
        by_impl[row['implementation']][shape_label(row)] = format(row[metric_key], value_format)

    implementation_width = max(len('implementation'), max(len(name) for name in implementations))
    column_widths = {
        label: max(len(label), max(len(by_impl[name].get(label, 'n/a')) for name in implementations))
        for label in shapes
    }

    header = '  '.join(
        [f"{'implementation':<{implementation_width}}"]
        + [f"{label:>{column_widths[label]}}" for label in shapes]
    )
    print(metric_label)
    print(header)
    print('-' * len(header))
    for name in implementations:
        row_values = [f"{name:<{implementation_width}}"]
        row_values.extend(
            f"{by_impl[name].get(label, 'n/a'):>{column_widths[label]}}" for label in shapes
        )
        print('  '.join(row_values))


def format_results(rows):
    print_metric_table(rows, 'median_ms', 'Median runtime (ms)', '.3f')
    print()
    print_metric_table(rows, 'peak_mb', 'Peak CUDA memory (MB)', '.2f')


The cell below runs the full benchmark sweep. Runtime is reported as the median of repeated runs after warmup. Memory is reported as the peak additional CUDA memory allocated during a single forward pass.


In [ ]:
results = []

for batch, heads, seq_len, dim in SHAPES:
    q, k, v, sm_scale = make_inputs(batch, heads, seq_len, dim)
    print(f"Running shape={(batch, heads, seq_len, dim)}")
    for name, fn in IMPLEMENTATIONS.items():
        median_ms = benchmark_runtime(fn, q, k, v, sm_scale)
        peak_mb = benchmark_peak_memory(fn, q, k, v, sm_scale)
        results.append(
            {
                "implementation": name,
                "B": batch,
                "H": heads,
                "S": seq_len,
                "D": dim,
                "median_ms": median_ms,
                "peak_mb": peak_mb,
            }
        )

format_results(results)
